# SpectraRestore — Google Colab

**KLA PS01 · SEMICON India Hackathon 2026**  
Joint denoise + 2× super-resolution (NAFNet-SR2×)

### Before you start
1. **Runtime → Change runtime type → T4 GPU** (or A100/L4 if available)
2. Put the project on Drive **or** upload the zip (Cell 2)
3. Put the KLA dataset on Drive under `MyDrive/SpectraRestore/data/` (Cell 3)

## 0 · Check GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Get the project code

Pick **one** option and run that cell only.

| Option | When to use |
|---|---|
| **A** | You uploaded `SpectraRestore.zip` to Drive |
| **B** | You already copied the whole project folder to Drive |
| **C** | You pushed the repo to GitHub |

In [ ]:
# === OPTION A: unzip from Drive ===
# Upload SpectraRestore.zip to MyDrive/SpectraRestore/ first
import zipfile
from pathlib import Path

ZIP = Path('/content/drive/MyDrive/SpectraRestore/SpectraRestore.zip')  # change if needed
PROJECT = Path('/content/SpectraRestore')

PROJECT.mkdir(parents=True, exist_ok=True)
assert ZIP.is_file(), f'Zip not found: {ZIP}\nUpload it to Drive first.'
with zipfile.ZipFile(ZIP, 'r') as z:
    z.extractall(PROJECT)

# if zip contained a nested folder, use that
inners = [p for p in PROJECT.iterdir() if p.is_dir() and (p / 'src').is_dir()]
if inners:
    PROJECT = inners[0]

%cd {PROJECT}
!ls -la
print('PROJECT =', PROJECT)

In [ ]:
# === OPTION B: project already on Drive ===
# Expected: MyDrive/SpectraRestore/{src,evaluate.py,requirements.txt,...}
from pathlib import Path
import shutil

SRC = Path('/content/drive/MyDrive/SpectraRestore')
PROJECT = Path('/content/SpectraRestore')

assert (SRC / 'src' / 'model.py').is_file(), f'Code not found at {SRC}'
if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(SRC, PROJECT, ignore=shutil.ignore_patterns(
    '.venv', '__pycache__', '.git', 'weights', 'outputs', 'data', '.firecrawl', 'slides'
))
%cd {PROJECT}
!ls -la
print('PROJECT =', PROJECT)

In [ ]:
# === OPTION C: clone from GitHub ===
# Replace with YOUR repo URL after you push
REPO_URL = 'https://github.com/<you>/<SpectraRestore>.git'  # ← edit me

%cd /content
!rm -rf SpectraRestore
!git clone {REPO_URL} SpectraRestore
%cd /content/SpectraRestore
!ls -la
PROJECT = '/content/SpectraRestore'
print('PROJECT =', PROJECT)

## 3 · Install dependencies

In [ ]:
# Colab already has torch+cuda; install the rest
!pip -q install lpips Pillow PyYAML tqdm gdown

import sys
from pathlib import Path
PROJECT = Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.model import build_model
m = build_model('default')
print(f"Model OK — {m.num_params()/1e6:.2f}M params")

## 4 · Dataset setup

### Recommended layout on Drive
```
MyDrive/SpectraRestore/data/
  train/
    degraded/   *.png / *.tif / *.npy
    gt/         same filenames
  val/
    degraded/
    gt/
```

Official KLA Drive folder:  
https://drive.google.com/drive/folders/1VKiFW-kDk9-q5XRPu3nrl08OM94EwzV6

Download it into `MyDrive/SpectraRestore/data/` (or symlink — Cell below).

In [ ]:
from pathlib import Path
import os

# Point this at wherever you put the KLA data on Drive
DRIVE_DATA = Path('/content/drive/MyDrive/SpectraRestore/data')
LOCAL_DATA = Path('/content/SpectraRestore/data')

LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)
if LOCAL_DATA.exists() or LOCAL_DATA.is_symlink():
    if LOCAL_DATA.is_symlink() or LOCAL_DATA.is_file():
        LOCAL_DATA.unlink()
    else:
        import shutil
        # keep local if it already has pairs; otherwise replace
        pass

if not LOCAL_DATA.exists():
    assert DRIVE_DATA.exists(), (
        f'Dataset not found at {DRIVE_DATA}\n'
        'Download the KLA folder into MyDrive/SpectraRestore/data/ first.'
    )
    # symlink so training reads from Drive without copying GBs
    os.symlink(DRIVE_DATA, LOCAL_DATA)

print('data ->', LOCAL_DATA.resolve())
!find -L data -type f | head -20
print('--- file counts ---')
!find -L data -type f | sed 's|/[^/]*$||' | sort | uniq -c | sort -rn | head

In [ ]:
# Optional: if the KLA zip is a shared Drive file/folder ID, pull with gdown
# Uncomment and set FOLDER_ID after you confirm the share link works for you.

# import gdown
# FOLDER_ID = '1VKiFW-kDk9-q5XRPu3nrl08OM94EwzV6'
# gdown.download_folder(id=FOLDER_ID, output='/content/drive/MyDrive/SpectraRestore/data', quiet=False)

## 5 · Quick smoke test (30 seconds)

In [ ]:
!python scripts/smoke_test.py

## 6 · Train

Tips for Colab free/T4:
- Start with `--preset default --batch_size 4` (or `8` if VRAM allows)
- Use fewer iters for a first pass (`50000`), then resume for full `200000`
- Weights are also copied to Drive so a disconnect doesn’t wipe them

In [ ]:
from pathlib import Path

PRESET = 'default'       # default (~29M) | fast (~15M) | large (~65M)
BATCH = 4                # raise to 8 on A100/L4
ITERS = 50000            # first pass; bump to 200000 for final
GT_CROP = 256

WEIGHTS_LOCAL = Path('weights')
WEIGHTS_DRIVE = Path('/content/drive/MyDrive/SpectraRestore/weights')
WEIGHTS_LOCAL.mkdir(exist_ok=True)
WEIGHTS_DRIVE.mkdir(parents=True, exist_ok=True)

# resume if a previous run exists on Drive
resume = ''
for name in ('last_ema.pt', 'best.pt'):
    cand = WEIGHTS_DRIVE / name
    if cand.is_file():
        # last_ema may not have optim; prefer latest ckpt_* if present
        break
ckpts = sorted(WEIGHTS_DRIVE.glob('ckpt_*.pt'))
if ckpts:
    resume = f' --resume {ckpts[-1]}'
    print('Resuming from', ckpts[-1])

cmd = f'''python -m src.train \
  --data_root data \
  --preset {PRESET} \
  --batch_size {BATCH} \
  --iters {ITERS} \
  --gt_crop {GT_CROP} \
  --num_workers 2 \
  --val_every 1000 \
  --save_every 2000 \
  --log_every 50 \
  --out_dir weights{resume}
'''
print(cmd)
!{cmd}

In [ ]:
# Mirror checkpoints to Drive (run after training / periodically)
import shutil
from pathlib import Path

src = Path('weights')
dst = Path('/content/drive/MyDrive/SpectraRestore/weights')
dst.mkdir(parents=True, exist_ok=True)
for f in src.glob('*.pt'):
    shutil.copy2(f, dst / f.name)
    print('copied', f.name)
!ls -lh /content/drive/MyDrive/SpectraRestore/weights | head

## 7 · Evaluate (KLA script)

Points `evaluate.py` at a folder of degraded images and writes restored outputs using the **same filenames** as the inputs.


In [ ]:
from pathlib import Path
import shutil

# Prefer Drive weights if local missing
for name in ('best.pt', 'last_ema.pt'):
    drive_w = Path('/content/drive/MyDrive/SpectraRestore/weights') / name
    local_w = Path('weights') / name
    if drive_w.is_file() and not local_w.is_file():
        Path('weights').mkdir(exist_ok=True)
        shutil.copy2(drive_w, local_w)
        print('restored', name, 'from Drive')

# Change INPUT to your test degraded folder
INPUT = 'data/val/degraded'          # or a KLA-released test folder
OUTPUT = 'outputs/val_restored'

# Outputs keep the SAME filenames as inputs (KLA scoring safety)
!python evaluate.py --input_dir {INPUT} --output_dir {OUTPUT} --weights weights/best.pt

# Optional: copy results to Drive
drive_out = Path('/content/drive/MyDrive/SpectraRestore/outputs')
drive_out.mkdir(parents=True, exist_ok=True)
!cp -r {OUTPUT} {drive_out}/
print('Saved to', drive_out)


## 8 · Preview a few restorations

In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

out_dir = Path('outputs/val_restored')
deg_dir = Path('data/val/degraded')
gt_dir = Path('data/val/gt')

# evaluate.py now writes the SAME filenames as inputs
restored = sorted([p for p in out_dir.rglob('*') if p.is_file()])
assert restored, f'No outputs in {out_dir} — run the evaluate cell first'
sample = random.choice(restored)
stem = sample.stem

def load(p):
    if p.suffix == '.npy':
        a = np.load(p)
    else:
        a = np.asarray(Image.open(p)).astype(np.float32)
        if a.max() > 1.5:
            a = a / 255.0
    if a.ndim == 3:
        a = a[..., 0]
    return np.clip(a, 0, 1)

deg_path = next(deg_dir.glob(stem + '.*'), None)
gt_path = next(gt_dir.glob(stem + '.*'), None) if gt_dir.exists() else None

fig, axs = plt.subplots(1, 3 if gt_path else 2, figsize=(12, 4))
axs[0].imshow(load(deg_path), cmap='gray'); axs[0].set_title('Degraded'); axs[0].axis('off')
axs[1].imshow(load(sample), cmap='gray'); axs[1].set_title('Restored'); axs[1].axis('off')
if gt_path:
    axs[2].imshow(load(gt_path), cmap='gray'); axs[2].set_title('GT'); axs[2].axis('off')
plt.tight_layout(); plt.show()
print(sample)


## Colab disconnect survival

1. Always run the **copy weights to Drive** cell after training chunks  
2. Re-open this notebook → Mount Drive → Option B → Install → set `resume` via existing `ckpt_*.pt`  
3. Keep the browser tab awake (or use Colab Pro) for long runs

Full design notes: see `SOLUTION.md` in the project.